# Tutorial: Basic STREAM data access

This tutorial introduces the public `StreamClient` interface. It shows how to discover sites, read a small water-quality and streamflow subset, retrieve catchment attributes, and combine the results.

The client will request your HydroShare credentials when it is created. This notebook reads remote Parquet data without downloading the complete STREAM resource.

## 1. Connect to HydroShare

In [ ]:
from streamwq.client import StreamClient
from streamwq.joins import join_timeseries

stream = StreamClient()

List the dataset names recognized by the client.

In [ ]:
stream.list_datasets()

## 2. Find suitable monitoring sites

Find sites that provide both **water temperature** and **dissolved oxygen**. A site must contain every requested variable.

In [ ]:
sites = stream.find_sites(variables=["WTemp_C", "DO_mgL"])

print(f"Found {len(sites):,} matching sites")
sites.head()

For a reproducible example, use a site known to contain these variables and **streamflow**. If that site is unavailable in a future resource version, use the first matching site instead.

In [ ]:
example_site = "STREAM-gauge-22"

if sites.empty:
    raise ValueError("No sites contain all requested variables")
if example_site not in sites["STREAM_ID"].values:
    example_site = sites.iloc[0]["STREAM_ID"]

print(f"Conditioned matched, using {example_site} for the rest of the tutorial")

## 3. Read water-quality observations

First, retrieve the complete water-quality record for the selected site and variables. We can then examine the earliest and latest timestamps to determine the period when observations are available.

In [ ]:
water_quality = stream.read_water_quality(
    site_ids=example_site,
    variables=["WTemp_C", "DO_mgL"]
)

water_quality

Read one month rather than the full record. The query selects the station partition, date range, variables, and approved quality flags before converting the result to pandas.

In [ ]:
water_quality = stream.read_water_quality(
    site_ids=example_site,
    variables=["WTemp_C", "DO_mgL"],
    start="2022-02-10",
    end="2022-03-10 23:59:59",
    quality="approved",
)

print(f"Retrieved {len(water_quality):,} water-quality records")
water_quality.head()

## 4. Read streamflow observations

Use the same station, dates, and quality policy so the two hourly datasets can be aligned.

In [ ]:
streamflow = stream.read_streamflow(
    site_ids=example_site,
    start="2022-02-10",
    end="2022-03-10 23:59:59",
    quality="approved",
)

print(f"Retrieved {len(streamflow):,} streamflow records")
streamflow.head()

## 5. Read static catchment attributes

Static attributes have one row per site. Here we request stream order, mean elevation, forest cover, and urban cover.

In [ ]:
attributes = stream.read_static_attributes(
    site_ids=example_site,
    columns=["stream_order", "ele_mt_sav", "for_pc_sse", "urb_pc_sse"],
)

attributes

## 6. Combine the datasets

Match hourly water quality and streamflow using exact `STREAM_ID` and `DateTime` values, then attach the site-level catchment attributes. `join_timeseries()` uses pandas internally but first protects against missing keys and duplicate station/timestamp rows that could silently produce an incorrect many-to-many merge. It also gives a clear error when periods do not overlap or no timestamps match exactly.

In [ ]:
combined_timeseries = join_timeseries(
    water_quality,
    streamflow,
    suffixes=("_wq", "_flow"),
)

combined = combined_timeseries.merge(
    attributes,
    on="STREAM_ID",
    how="left",
)

print(f"Combined dataset contains {len(combined):,} rows")
combined.head()

## 7. Inspect and optionally export the subset

In [ ]:
combined[["WTemp_C", "DO_mgL", "Q_m3s"]].describe()

In [ ]:
# Uncomment to save the analysis-ready subset.
# combined.to_parquet("stream_basic_subset.parquet", index=False)

## Next steps

You can change the site filters, variables, time period, and accepted quality flags while keeping the same workflow. Later tutorials will cover spatial selection, temporal summaries, and event-based analysis.